# Project 5 — Complete & Simple Jupyter Notebook
## User Analytics in the Telecommunication Industry

**Purpose:** Complete the supplied Project 5 brief using:
1. `telcom_data (2).xlsx` — telecom xDR dataset
2. `Field Descriptions.xlsx` — data dictionary
3. `project 5.pdf` — project requirements

The notebook is written in **simple, step-by-step Python**. Every major requirement from Tasks 1–4 is included, plus reusable cleaning code, feature store, MySQL export code, model tracking, Docker, CI/CD, tests and a dashboard starter.




## 0. Project Criteria Covered

| Requirement | Notebook section |
|---|---|
| Data dictionary / variable types | Section 1 |
| Missing values + outliers | Section 2 |
| Top 10 handsets | Task 1 |
| Top 3 manufacturers | Task 1 |
| Top 5 handsets per manufacturer | Task 1 |
| User/application aggregation | Task 1 |
| Basic statistics | Task 1 |
| Univariate analysis | Task 1 |
| Graphical analysis | Task 1 |
| Bivariate analysis | Task 1 |
| Duration deciles | Task 1 |
| Correlation | Task 1 |
| PCA | Task 1 |
| Top 10 engagement customers | Task 2 |
| K-Means k=3 | Task 2 |
| Cluster metrics | Task 2 |
| Top users by application | Task 2 |
| Top 3 applications | Task 2 |
| Elbow method / optimized k | Task 2 |
| TCP / RTT / Throughput | Task 3 |
| Top/bottom/frequent values | Task 3 |
| Handset experience analysis | Task 3 |
| Experience K-Means k=3 | Task 3 |
| Engagement score | Task 4 |
| Experience score | Task 4 |
| Satisfaction score | Task 4 |
| Top 10 satisfied customers | Task 4 |
| Satisfaction regression | Task 4 |
| Satisfaction K-Means k=2 | Task 4 |
| MySQL export | Section 3 |
| Model tracking | Section 4 |
| Dashboard / Docker / CI / tests | Section 5 |


In [ ]:
# 1. Install libraries if needed
# Run this cell only if a package is missing.

# %pip install pandas numpy matplotlib seaborn scikit-learn openpyxl joblib mysql-connector-python streamlit

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

DATA_FILE = "telcom_data (2).xlsx"
FIELD_FILE = "Field Descriptions.xlsx"

os.makedirs("outputs", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("deployment", exist_ok=True)

print("Project folder:", os.getcwd())


In [ ]:
# 2. Read the data dictionary first
# This helps us understand the variables before analysis.

field_df = pd.read_excel(FIELD_FILE)
print("Field description shape:", field_df.shape)
display(field_df.head(10))
print("\nField description columns:")
print(field_df.columns.tolist())


In [ ]:
# 2.1 Read telecom data
# The original file is large, so Excel reading can take some time.

data = pd.read_excel(DATA_FILE)

print("Dataset shape:", data.shape)
print("\nFirst 5 rows:")
display(data.head())

print("\nColumn names:")
for i, col in enumerate(data.columns, 1):
    print(i, col)


## 3. Data Quality: Missing Values and Outliers

The project brief specifically asks us to:
- identify missing values;
- identify outliers;
- replace missing numerical values with the column mean;
- replace numerical outliers with the column mean;
- use mode for categorical values where appropriate.

The functions below are reusable and can be used again on a similar telecom dataset.


In [ ]:
# 3.1 Helper functions

def find_column(df, possible_names):
    """Return the first matching column name from a list of possible names."""
    for name in possible_names:
        if name in df.columns:
            return name
    return None


def clean_data(df):
    """Fill missing values and replace IQR outliers with the mean."""
    df = df.copy()

    # Numerical columns
    numeric_cols = df.select_dtypes(include=np.number).columns

    for col in numeric_cols:
        mean_value = df[col].mean()
        df[col] = df[col].fillna(mean_value)

        # IQR outlier rule
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        if iqr > 0:
            low = q1 - 1.5 * iqr
            high = q3 + 1.5 * iqr
            outlier_mask = (df[col] < low) | (df[col] > high)
            df.loc[outlier_mask, col] = mean_value

    # Categorical columns
    categorical_cols = df.select_dtypes(exclude=np.number).columns

    for col in categorical_cols:
        if df[col].isna().any():
            mode = df[col].mode()
            if len(mode) > 0:
                df[col] = df[col].fillna(mode.iloc[0])
            else:
                df[col] = df[col].fillna("Unknown")

    return df


# Missing values before cleaning
missing_before = data.isnull().sum()
missing_before = missing_before[missing_before > 0].sort_values(ascending=False)

print("Columns containing missing values:")
display(missing_before.to_frame("Missing Values"))

clean = clean_data(data)

print("Total missing values after cleaning:", clean.isnull().sum().sum())


In [ ]:
# 3.2 Data types and basic statistics

data_types = pd.DataFrame({
    "Column": clean.columns,
    "Data Type": clean.dtypes.astype(str).values,
    "Missing Values": clean.isnull().sum().values
})

display(data_types)

print("Basic numerical statistics:")
display(clean.describe().T)


# Task 1 — User Overview Analysis

This section follows the Project 5 requirements for handset analysis, customer/application aggregation, EDA, deciles, correlation and PCA.


In [ ]:
# 4. Top 10 handsets

handset_col = find_column(clean, ["Handset Type", "Handset"])

top10_handsets = (
    clean[handset_col]
    .value_counts()
    .head(10)
    .reset_index()
)
top10_handsets.columns = ["Handset", "Count"]

display(top10_handsets)

plt.figure(figsize=(10, 5))
sns.barplot(data=top10_handsets, x="Count", y="Handset")
plt.title("Top 10 Handsets")
plt.tight_layout()
plt.show()


In [ ]:
# 4.1 Top 3 handset manufacturers

manufacturer_col = find_column(
    clean,
    ["Handset Manufacturer", "Handset Manufacturer "]
)

top3_manufacturers = clean[manufacturer_col].value_counts().head(3).reset_index()
top3_manufacturers.columns = ["Manufacturer", "Count"]

display(top3_manufacturers)


In [ ]:
# 4.2 Top 5 handsets for each of the top 3 manufacturers

top3_names = top3_manufacturers["Manufacturer"].tolist()

top5_each = (
    clean[clean[manufacturer_col].isin(top3_names)]
    .groupby([manufacturer_col, handset_col])
    .size()
    .reset_index(name="Count")
    .sort_values([manufacturer_col, "Count"], ascending=[True, False])
    .groupby(manufacturer_col)
    .head(5)
)

display(top5_each)


### Marketing interpretation

Use the tables above to identify:
- the most common devices;
- dominant manufacturers;
- device combinations that deserve compatibility/testing priority;
- possible handset-specific promotions.

The recommendation should be based on the observed counts rather than assumptions.


In [ ]:
# 4.3 Create application traffic columns
# xDR data is tracked for Social Media, Google, Email, YouTube, Netflix, Gaming and Other.

app_columns = {
    "Social Media": ["Social Media DL (Bytes)", "Social Media UL (Bytes)"],
    "Google": ["Google DL (Bytes)", "Google UL (Bytes)"],
    "Email": ["Email DL (Bytes)", "Email UL (Bytes)"],
    "YouTube": ["Youtube DL (Bytes)", "Youtube UL (Bytes)"],
    "Netflix": ["Netflix DL (Bytes)", "Netflix UL (Bytes)"],
    "Gaming": ["Gaming DL (Bytes)", "Gaming UL (Bytes)"],
    "Other": ["Other DL (Bytes)", "Other UL (Bytes)"]
}

# Make names flexible for small spelling differences.
def locate_application_columns(df, candidates):
    result = []
    for c in candidates:
        if c in df.columns:
            result.append(c)
    return result

app_found = {}
for app, candidates in app_columns.items():
    found = locate_application_columns(clean, candidates)
    app_found[app] = found
    print(app, "->", found)

for app, cols in app_found.items():
    if len(cols) == 2:
        clean[app + "_data"] = clean[cols].sum(axis=1)


In [ ]:
# 4.4 Aggregate user-level information

user_col = find_column(
    clean,
    ["MSISDN/Number", "MSISDN", "MSISDN/Number "]
)

duration_col = find_column(
    clean,
    ["Dur. (ms)", "Dur (ms)", "Duration"]
)

dl_col = find_column(clean, ["Total DL (Bytes)", "Total DL"])
ul_col = find_column(clean, ["Total UL (Bytes)", "Total UL"])

# Create total traffic if not already present
clean["total_traffic"] = clean[dl_col] + clean[ul_col]

grouped = clean.groupby(user_col)

user = grouped.agg(
    sessions=(user_col, "size"),
    total_duration_ms=(duration_col, "sum"),
    total_dl=(dl_col, "sum"),
    total_ul=(ul_col, "sum"),
    total_traffic=("total_traffic", "sum")
)

# Add application totals
for app in app_found:
    col = app + "_data"
    if col in clean.columns:
        user[col] = grouped[col].sum()

user = user.reset_index()

print("Number of unique customers:", user.shape[0])
display(user.head())


In [ ]:
# 4.5 Non-graphical univariate analysis

numeric_user_cols = [
    "sessions", "total_duration_ms", "total_dl",
    "total_ul", "total_traffic"
]

for app in app_found:
    if app + "_data" in user.columns:
        numeric_user_cols.append(app + "_data")

summary = user[numeric_user_cols].describe().T
summary["variance"] = user[numeric_user_cols].var()
summary["std"] = user[numeric_user_cols].std()

display(summary[["count", "mean", "50%", "std", "variance", "min", "max"]])


In [ ]:
# 4.6 Graphical univariate analysis

# Distribution of sessions
plt.figure(figsize=(8, 4))
sns.histplot(user["sessions"], bins=30)
plt.title("Distribution of Sessions per Customer")
plt.xlabel("Number of sessions")
plt.show()

# Distribution of total traffic
plt.figure(figsize=(8, 4))
sns.histplot(user["total_traffic"], bins=30)
plt.title("Distribution of Total Traffic per Customer")
plt.xlabel("Total DL + UL bytes")
plt.show()


In [ ]:
# 4.7 Bivariate analysis: each application vs total traffic

relationship = []

for app in app_found:
    col = app + "_data"
    if col in user.columns:
        correlation = user[col].corr(user["total_traffic"])
        relationship.append([app, correlation])

relationship = pd.DataFrame(
    relationship,
    columns=["Application", "Correlation with Total Traffic"]
).sort_values("Correlation with Total Traffic", ascending=False)

display(relationship)

# Scatter plots for applications
for app in app_found:
    col = app + "_data"
    if col in user.columns:
        plt.figure(figsize=(6, 4))
        plt.scatter(user[col], user["total_traffic"], alpha=0.25)
        plt.xlabel(app + " traffic")
        plt.ylabel("Total traffic")
        plt.title(app + " vs Total Traffic")
        plt.tight_layout()
        plt.show()


In [ ]:
# 4.8 Duration deciles — top five decile classes

user["duration_decile"] = pd.qcut(
    user["total_duration_ms"].rank(method="first"),
    10,
    labels=False
) + 1

decile_table = (
    user.groupby("duration_decile")
    .agg(
        customers=("total_traffic", "size"),
        total_data=("total_traffic", "sum")
    )
    .reset_index()
)

decile_table["traffic_share_percent"] = (
    decile_table["total_data"] /
    decile_table["total_data"].sum() * 100
)

print("Top five duration deciles:")
display(decile_table.sort_values("duration_decile", ascending=False).head(5))


In [ ]:
# 4.9 Correlation matrix for the seven applications

application_data_cols = [
    app + "_data" for app in app_found
    if app + "_data" in user.columns
]

correlation_matrix = user[application_data_cols].corr()

display(correlation_matrix.round(3))

plt.figure(figsize=(9, 7))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f")
plt.title("Application Traffic Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# 4.10 PCA — dimensionality reduction

X_app = user[application_data_cols].fillna(0)

scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_app)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

pca_table = pd.DataFrame({
    "Principal Component": range(1, len(pca.explained_variance_ratio_) + 1),
    "Explained Variance": pca.explained_variance_ratio_,
    "Cumulative Variance": np.cumsum(pca.explained_variance_ratio_)
})

display(pca_table)

plt.figure(figsize=(8, 4))
plt.plot(
    pca_table["Principal Component"],
    pca_table["Cumulative Variance"],
    marker="o"
)
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Explained Variance")
plt.show()

loadings = pd.DataFrame(
    pca.components_[:2].T,
    index=application_data_cols,
    columns=["PC1", "PC2"]
)

print("PCA loadings:")
display(loadings)


### PCA interpretation — maximum 4 points

After running the cell above, use the largest absolute loadings and cumulative variance to write four concise points:
1. Which applications contribute most to PC1?
2. Which applications contribute most to PC2?
3. How much variance is explained by the first 2–3 components?
4. How can these components simplify customer segmentation?


# Task 2 — User Engagement Analysis

Engagement is measured using:
- session frequency;
- session duration;
- total session traffic (DL + UL).


In [ ]:
# 5.1 Top 10 customers by each engagement metric

engagement_metrics = ["sessions", "total_duration_ms", "total_traffic"]

for metric in engagement_metrics:
    print("\nTop 10 customers by:", metric)
    display(
        user[[user_col, metric]]
        .sort_values(metric, ascending=False)
        .head(10)
    )


In [ ]:
# 5.2 Normalize engagement metrics and K-Means k=3

engagement_X = user[engagement_metrics].fillna(0)

engagement_scaler = StandardScaler()
engagement_scaled = engagement_scaler.fit_transform(engagement_X)

engagement_k3 = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

user["engagement_cluster"] = engagement_k3.fit_predict(engagement_scaled)

print("Cluster counts:")
display(user["engagement_cluster"].value_counts().sort_index())

# Cluster summary in original units
engagement_summary = (
    user.groupby("engagement_cluster")[engagement_metrics]
    .agg(["min", "max", "mean", "sum"])
)

display(engagement_summary)


In [ ]:
# 5.3 Elbow method — choose an optimized k

inertia = []

for k in range(2, 10):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(engagement_scaled)
    inertia.append(model.inertia_)

elbow_table = pd.DataFrame({
    "k": range(2, 10),
    "inertia": inertia
})

display(elbow_table)

plt.figure(figsize=(7, 4))
plt.plot(elbow_table["k"], elbow_table["inertia"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Engagement")
plt.show()

print("The optimized k should be selected from the visible elbow in the chart.")


In [ ]:
# 5.4 Top 10 users per application

for app in app_found:
    col = app + "_data"
    if col in user.columns:
        print("\nTop 10 users for", app)
        display(
            user[[user_col, col]]
            .sort_values(col, ascending=False)
            .head(10)
        )

# Top 3 applications by total traffic
application_totals = (
    user[application_data_cols]
    .sum()
    .sort_values(ascending=False)
)

print("Applications ranked by total traffic:")
display(application_totals.to_frame("Total Traffic"))

top3_apps = application_totals.head(3)

plt.figure(figsize=(8, 4))
top3_apps.plot(kind="bar")
plt.title("Top 3 Most Used Applications")
plt.ylabel("Total Traffic")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Task 3 — User Experience Analysis

Experience uses:
- TCP retransmission;
- RTT;
- throughput;
- handset type.

For missing/outlier treatment, the reusable cleaning function from Section 3 is applied before aggregation.


In [ ]:
# 6.1 Locate experience columns

tcp_col = find_column(clean, [
    "TCP DL Retrans. Vol (Bytes)",
    "TCP DL Retransmission (Bytes)",
    "TCP Retrans. Vol (Bytes)"
])

rtt_col = find_column(clean, [
    "Avg RTT DL (ms)",
    "Avg RTT (ms)"
])

throughput_col = find_column(clean, [
    "Avg Bearer TP DL (kbps)",
    "Avg Bearer TP DL (kbps) "
])

print("TCP column:", tcp_col)
print("RTT column:", rtt_col)
print("Throughput column:", throughput_col)


In [ ]:
# 6.2 Aggregate experience metrics per customer

experience = clean.groupby(user_col).agg(
    average_tcp=(tcp_col, "mean"),
    average_rtt=(rtt_col, "mean"),
    average_throughput=(throughput_col, "mean")
).reset_index()

# Add most common handset for each customer
customer_handset = (
    clean.groupby(user_col)[handset_col]
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) else "Unknown")
    .reset_index(name="handset_type")
)

experience = experience.merge(customer_handset, on=user_col, how="left")

user = user.merge(experience, on=user_col, how="left")

display(user.head())


In [ ]:
# 6.3 Top, bottom and most frequent values

for col in [tcp_col, rtt_col, throughput_col]:

    print("\n==============================")
    print(col)

    print("Top 10:")
    display(clean[[col]].sort_values(col, ascending=False).head(10))

    print("Bottom 10:")
    display(clean[[col]].sort_values(col, ascending=True).head(10))

    print("Most frequent values:")
    display(clean[col].value_counts().head(10).to_frame("Frequency"))


In [ ]:
# 6.4 Average throughput per handset type

throughput_handset = (
    user.groupby("handset_type")["average_throughput"]
    .mean()
    .sort_values(ascending=False)
)

display(throughput_handset.head(20).to_frame("Average Throughput"))

plt.figure(figsize=(10, 6))
throughput_handset.head(15).plot(kind="bar")
plt.title("Average Throughput by Handset Type — Top 15")
plt.ylabel("Average Throughput (kbps)")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()


In [ ]:
# 6.5 Average TCP retransmission per handset type

tcp_handset = (
    user.groupby("handset_type")["average_tcp"]
    .mean()
    .sort_values(ascending=False)
)

display(tcp_handset.head(20).to_frame("Average TCP Retransmission"))

plt.figure(figsize=(10, 6))
tcp_handset.head(15).plot(kind="bar")
plt.title("Average TCP Retransmission by Handset Type — Top 15")
plt.ylabel("Average TCP Retransmission")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()


In [ ]:
# 6.6 Experience K-Means with k=3

experience_metrics = [
    "average_tcp",
    "average_rtt",
    "average_throughput"
]

experience_X = user[experience_metrics].fillna(
    user[experience_metrics].mean()
)

experience_scaler = StandardScaler()
experience_scaled = experience_scaler.fit_transform(experience_X)

experience_k3 = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

user["experience_cluster"] = experience_k3.fit_predict(experience_scaled)

experience_cluster_summary = (
    user.groupby("experience_cluster")[experience_metrics]
    .agg(["min", "max", "mean", "sum"])
)

display(experience_cluster_summary)

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=user,
    x="average_throughput",
    y="average_rtt",
    hue="experience_cluster",
    palette="deep"
)
plt.title("Experience Clusters")
plt.show()


### Experience cluster interpretation

A practical interpretation is:
- **Better experience:** higher throughput with lower RTT and lower retransmission.
- **Poorer experience:** lower throughput with higher RTT and/or higher retransmission.
- **Intermediate experience:** values between the two groups.

The actual cluster labels are arbitrary, so the cluster means above—not the numeric label itself—must be used to describe each group.


# Task 4 — User Satisfaction Analysis

The project defines:
- **Engagement score:** Euclidean distance from the less-engaged cluster.
- **Experience score:** Euclidean distance from the worst-experience cluster.
- **Satisfaction score:** average of engagement and experience scores.

Because K-Means labels are arbitrary, the notebook identifies the relevant reference clusters from their actual metric values.


In [ ]:
# 7.1 Identify reference clusters

# Less engaged = cluster with the smallest average normalized engagement level.
engagement_centers = pd.DataFrame(
    engagement_k3.cluster_centers_,
    columns=engagement_metrics
)

less_engaged_cluster = engagement_centers.sum(axis=1).idxmin()

# For experience:
# lower throughput + higher RTT + higher TCP retransmission = poorer experience.
experience_centers = pd.DataFrame(
    experience_k3.cluster_centers_,
    columns=experience_metrics
)

experience_badness = (
    experience_centers["average_tcp"]
    + experience_centers["average_rtt"]
    - experience_centers["average_throughput"]
)

worst_experience_cluster = experience_badness.idxmax()

print("Less engaged cluster:", less_engaged_cluster)
print("Worst experience cluster:", worst_experience_cluster)


In [ ]:
# 7.2 Calculate engagement and experience scores

# Distance calculations use standardized variables.
engagement_reference = engagement_k3.cluster_centers_[less_engaged_cluster]

user_engagement_scaled = engagement_scaler.transform(
    user[engagement_metrics]
)

user["engagement_score"] = np.linalg.norm(
    user_engagement_scaled - engagement_reference,
    axis=1
)

experience_reference = experience_k3.cluster_centers_[worst_experience_cluster]

user_experience_scaled = experience_scaler.transform(
    user[experience_metrics]
)

user["experience_score"] = np.linalg.norm(
    user_experience_scaled - experience_reference,
    axis=1
)

# Satisfaction = average of both scores
user["satisfaction_score"] = (
    user["engagement_score"] +
    user["experience_score"]
) / 2

print("Scores created successfully.")
display(
    user[
        [user_col, "engagement_score",
         "experience_score", "satisfaction_score"]
    ].head(10)
)


In [ ]:
# 7.3 Top 10 satisfied customers

top10_satisfied = (
    user[
        [user_col, "engagement_score",
         "experience_score", "satisfaction_score"]
    ]
    .sort_values("satisfaction_score", ascending=False)
    .head(10)
)

display(top10_satisfied)


In [ ]:
# 7.4 Regression model to predict satisfaction score

regression_features = [
    "engagement_score",
    "experience_score",
    "sessions",
    "total_duration_ms",
    "total_traffic",
    "average_tcp",
    "average_rtt",
    "average_throughput"
]

model_data = user[regression_features + ["satisfaction_score"]].dropna()

X = model_data[regression_features]
y = model_data["satisfaction_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

regression_model = LinearRegression()
regression_model.fit(X_train, y_train)

prediction = regression_model.predict(X_test)

r2 = r2_score(y_test, prediction)
rmse = mean_squared_error(y_test, prediction) ** 0.5

print("R2 Score:", round(r2, 4))
print("RMSE:", round(rmse, 4))

coefficients = pd.DataFrame({
    "Feature": regression_features,
    "Coefficient": regression_model.coef_
})

display(coefficients.sort_values("Coefficient", key=abs, ascending=False))

plt.figure(figsize=(6, 5))
plt.scatter(y_test, prediction, alpha=0.3)
plt.xlabel("Actual Satisfaction")
plt.ylabel("Predicted Satisfaction")
plt.title("Satisfaction Regression")
plt.show()


In [ ]:
# 7.5 K-Means k=2 on engagement + experience scores

score_data = user[
    ["engagement_score", "experience_score"]
].fillna(0)

score_scaler = StandardScaler()
score_scaled = score_scaler.fit_transform(score_data)

satisfaction_k2 = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

user["satisfaction_cluster"] = satisfaction_k2.fit_predict(score_scaled)

satisfaction_summary = (
    user.groupby("satisfaction_cluster")
    [["engagement_score", "experience_score", "satisfaction_score"]]
    .mean()
)

display(satisfaction_summary)


## 7.6 Satisfaction interpretation

Compare the two satisfaction clusters using the displayed averages:
- the cluster with higher average scores has greater distance from the defined low-engagement/worst-experience reference points;
- the numeric cluster ID itself has no business meaning;
- use the averages and supporting metrics when writing the final business interpretation.


# 8. Feature Store + MySQL Export

The project asks for a final table containing:
- user ID;
- engagement score;
- experience score;
- satisfaction score.

The next cell creates a reusable feature-store CSV. The MySQL section provides the exact export/query code required for the report screenshot.


In [ ]:
# 8.1 Create final feature store

final_scores = user[
    [
        user_col,
        "engagement_score",
        "experience_score",
        "satisfaction_score"
    ]
].copy()

final_scores.to_csv(
    "outputs/tables/final_customer_scores.csv",
    index=False
)

final_scores.to_csv(
    "outputs/tables/feature_store.csv",
    index=False
)

print("Feature store saved:")
print("outputs/tables/final_customer_scores.csv")
print("outputs/tables/feature_store.csv")

display(final_scores.head())


In [ ]:
# 8.2 MySQL export
# Fill in your own MySQL credentials before running.
# This cell is intentionally separate so the notebook still works without MySQL.

# import mysql.connector

# connection = mysql.connector.connect(
#     host="localhost",
#     user="root",
#     password="YOUR_PASSWORD",
#     database="tellco"
# )

# cursor = connection.cursor()

# cursor.execute("DROP TABLE IF EXISTS final_customer_scores")

# cursor.execute("""
# CREATE TABLE final_customer_scores (
#     user_id VARCHAR(100),
#     engagement_score DOUBLE,
#     experience_score DOUBLE,
#     satisfaction_score DOUBLE
# )
# """)

# for _, row in final_scores.iterrows():
#     cursor.execute(
#         """
#         INSERT INTO final_customer_scores
#         VALUES (%s, %s, %s, %s)
#         """,
#         tuple(row)
#     )

# connection.commit()

# cursor.execute(
#     "SELECT * FROM final_customer_scores LIMIT 10"
# )

# for row in cursor.fetchall():
#     print(row)

# cursor.close()
# connection.close()

print("MySQL code is ready. Add your local MySQL credentials and run it.")


# 9. Model Deployment Tracking / MLOps

The brief requires tracking:
- code version;
- start/end time;
- source;
- parameters;
- metrics;
- artifacts/output.

The following lightweight tracking file can be produced without requiring a paid MLOps service.


In [ ]:
# 9.1 Save model and tracking information

import joblib
from datetime import datetime

start_time = datetime.now().isoformat()

joblib.dump(
    regression_model,
    "deployment/satisfaction_regression_model.joblib"
)

tracking = {
    "model": "Linear Regression",
    "source": "TellCo Project 5 telecom dataset",
    "algorithm_parameters": {
        "test_size": 0.20,
        "random_state": 42
    },
    "metrics": {
        "R2": float(r2),
        "RMSE": float(rmse)
    },
    "reference_clusters": {
        "less_engaged_cluster": int(less_engaged_cluster),
        "worst_experience_cluster": int(worst_experience_cluster)
    },
    "artifact": "deployment/satisfaction_regression_model.joblib",
    "final_table": "outputs/tables/final_customer_scores.csv",
    "start_time": start_time,
    "end_time": datetime.now().isoformat()
}

with open("outputs/model_tracking.json", "w") as f:
    json.dump(tracking, f, indent=4)

display(pd.DataFrame([tracking]))


# 10. Dashboard, Tests, Docker and CI/CD

The Project 5 brief also asks for:
- a web-based dashboard;
- installable code;
- unit tests;
- CI/CD using GitHub Actions or Travis;
- Dockerfile.

The following cells create simple starter files from the notebook so the analytical logic remains easy to understand.


In [ ]:
# 10.1 Create requirements.txt

requirements = """pandas
numpy
matplotlib
seaborn
scikit-learn
openpyxl
joblib
streamlit
mysql-connector-python
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print(requirements)


In [ ]:
# 10.2 Create a simple Streamlit dashboard

dashboard_code = r'''
import streamlit as st
import pandas as pd

st.set_page_config(page_title="TellCo User Analytics", layout="wide")

st.title("TellCo User Analytics Dashboard")

scores = pd.read_csv("outputs/tables/final_customer_scores.csv")

st.subheader("Customer Scores")
st.dataframe(scores.head(20), use_container_width=True)

col1, col2, col3 = st.columns(3)

with col1:
    st.metric("Customers", len(scores))

with col2:
    st.metric("Average Engagement Score",
              round(scores["engagement_score"].mean(), 2))

with col3:
    st.metric("Average Satisfaction Score",
              round(scores["satisfaction_score"].mean(), 2))

st.subheader("Satisfaction Distribution")
st.bar_chart(
    scores["satisfaction_score"].value_counts(bins=10).sort_index()
)
'''

with open("dashboard_app.py", "w") as f:
    f.write(dashboard_code)

print("Dashboard created: dashboard_app.py")
print("Run with: streamlit run dashboard_app.py")


In [ ]:
# 10.3 Create a simple unit test

test_code = r'''
import pandas as pd

def test_final_score_columns():
    df = pd.read_csv("outputs/tables/final_customer_scores.csv")
    required = {
        "engagement_score",
        "experience_score",
        "satisfaction_score"
    }
    assert required.issubset(df.columns)

def test_scores_are_numeric():
    df = pd.read_csv("outputs/tables/final_customer_scores.csv")
    assert pd.api.types.is_numeric_dtype(df["satisfaction_score"])
'''

os.makedirs("tests", exist_ok=True)

with open("tests/test_project.py", "w") as f:
    f.write(test_code)

print("Unit test created: tests/test_project.py")


In [ ]:
# 10.4 Create Dockerfile

dockerfile = """FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

CMD [\"streamlit\", \"run\", \"dashboard_app.py\", \"--server.address=0.0.0.0\"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

print(dockerfile)


In [ ]:
# 10.5 Create GitHub Actions CI/CD test workflow

os.makedirs(".github/workflows", exist_ok=True)

workflow = """name: TellCo Project Tests

on:
  push:
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install -r requirements.txt pytest

      - name: Run tests
        run: pytest -q
"""

with open(".github/workflows/ci.yml", "w") as f:
    f.write(workflow)

print("GitHub Actions workflow created.")


# 11. Final Business Findings

The following cell automatically produces a compact summary after all analysis cells have been executed. It does **not** invent a recommendation; it uses the calculated values.

For the final report, explain:
1. customer/device overview;
2. engagement segments;
3. network experience segments;
4. satisfaction segments;
5. major applications driving traffic;
6. limitations such as one-month aggregated xDR data, data-quality treatment, clustering assumptions and lack of financial information in this analysis.

The project brief asks for a purchase recommendation. The final recommendation should be written after reviewing the calculated evidence below.


In [ ]:
# 11.1 Automatic summary for final report

print("========== FINAL PROJECT SUMMARY ==========")

print("Customers:", len(user))

print("\nTop 3 Applications:")
display(application_totals.head(3).to_frame("Total Traffic"))

print("\nEngagement cluster averages:")
display(
    user.groupby("engagement_cluster")[engagement_metrics].mean()
)

print("\nExperience cluster averages:")
display(
    user.groupby("experience_cluster")[experience_metrics].mean()
)

print("\nSatisfaction cluster averages:")
display(satisfaction_summary)

print("\nRegression R2:", round(r2, 4))
print("Regression RMSE:", round(rmse, 4))

print("\nTop 10 satisfied customers:")
display(top10_satisfied)


# 12. Submission Checklist

Before submission, verify that all cells have been executed and these files exist:

### Analysis
- [ ] `outputs/tables/final_customer_scores.csv`
- [ ] `outputs/tables/feature_store.csv`
- [ ] `outputs/model_tracking.json`
- [ ] figures generated by Task 1–4

### Engineering
- [ ] `requirements.txt`
- [ ] `tests/test_project.py`
- [ ] `.github/workflows/ci.yml`
- [ ] `Dockerfile`
- [ ] `dashboard_app.py`

### Report / presentation
- [ ] Task 1 findings
- [ ] Task 2 findings
- [ ] Task 3 findings
- [ ] Task 4 findings
- [ ] purchase/growth recommendation based on calculated evidence
- [ ] limitations
- [ ] dashboard screenshot
- [ ] MySQL SELECT screenshot
- [ ] model tracking output screenshot

**Note:** The supplied brief also asks for a written report and a presentation of no more than 20 slides. This notebook supplies the analytical evidence and artifacts for those deliverables.
